# Cluster Analysis

Differences between groups were tested using a Chi-squared test for categorical variables and a one-way ANOVA for continuous variables. 

In [4]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sn

from IPython.display import display

from preproc import get_hfpef_200, convert2np, get_hfpef_200

from scipy.stats import f_oneway, chi2_contingency

from sklearn.metrics.cluster import contingency_matrix

np.set_printoptions(precision=3, suppress=True)
pd.options.display.float_format = "{:,.3f}".format

In [5]:
data_df = get_hfpef_200()

## Get Cluster

In [12]:
from methods import get_sc_pred, get_p_value
from utils import get_score, score_columns, plot_clustering_score, plot_contingency_matrix, plot_bic_aic

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')

In [13]:
no_clusters = 3
lbl_colname= ['Death', 'CV death', 'Major cardiac events']

In [14]:
X, y, feature_list = convert2np(data_df, lbl_colname)

In [15]:
y_pred = get_sc_pred(2, X)

## Analysis

In [16]:
feat_df2 = get_p_value(data_df, y_pred)

In [17]:
feat_df = pd.DataFrame(index=data_df.columns)
feat_df['is_category'] = False

In [18]:
print('Continuous')
for col in data_df:
    if data_df[col].unique().shape[0] > 8:
        print(col, data_df[col].unique().shape)

print()
print('Category')
for col in data_df:
    if data_df[col].unique().shape[0] <= 8:
        data_df[col] = data_df[col].astype('int')
        feat_df.loc[col,'is_category'] = True
        print(col, data_df[col].unique().shape)

Continuous
Age (47,)
Cr (107,)
GFR (93,)
BMI (131,)
BSA (45,)
SBP (68,)
DBP (53,)
MAP (109,)
PP (69,)
1/2SBP (68,)
HR (58,)
NTProBNP (119,)
medial a' (109,)
medial E' (126,)
lateral E' (138,)
average e/E' (153,)
lateral e/E' (133,)
E/A (35,)
Mitral E/e' (134,)
TR Vmax (99,)
RWT (88,)
LV mass index (170,)
LAVI (164,)
LACI (112,)
LA diameter (108,)
LVEF (76,)

Category
Sex (2,)
CKD stage (5,)
smoke (2,)
DM (2,)
Insulin (2,)
AF (2,)
wide PP (2,)
ACE-i (2,)
ARB (2,)
ARNI (2,)
betablocker (2,)
Aspirin (2,)
SGLT2i (2,)
MRA (2,)
MI (2,)
stroke (2,)
COPD (2,)
NYHA (4,)
edema (2,)
Death (2,)
CV death (2,)
Major cardiac events (2,)
LVH (8,)


Compute p-value for continuous features

In [ ]:
feat_df['p-value'] = 0.0
cont_col = feat_df[~feat_df['is_category']].index
clusters = []
for yi in np.unique(y_pred):
    idx = y_pred == yi
    clusters.append(data_df.loc[idx, cont_col])
    # display(data_df.loc[idx, cont_col])
#     print(idx)

In [20]:
F, p = f_oneway(*clusters)
feat_df.loc[cont_col, 'p-value'] = p
feat_df

,is_category,p-value
Age,False,0.416
Sex,True,0.000
Cr,False,0.218
GFR,False,0.461
CKD stage,True,0.000
smoke,True,0.000
BMI,False,0.951
BSA,False,0.615
DM,True,0.000
Insulin,True,0.000


Compute p-value for category features

In [21]:
for cat_col in feat_df[feat_df['is_category']].index:
    print(cat_col)
    
    obs = contingency_matrix(y_pred, data_df[cat_col])
    chi2, p, dof, ex = chi2_contingency(obs.T, correction=False)
    feat_df.loc[cat_col,'p-value'] = p

Sex
CKD stage
smoke
DM
Insulin
AF
wide PP
ACE-i
ARB
ARNI
betablocker
Aspirin
SGLT2i
MRA
MI
stroke
COPD
NYHA
edema
Death
CV death
Major cardiac events
LVH


In [22]:
feat_df

,is_category,p-value
Age,False,0.416
Sex,True,0.457
Cr,False,0.218
GFR,False,0.461
CKD stage,True,0.387
smoke,True,0.425
BMI,False,0.951
BSA,False,0.615
DM,True,0.795
Insulin,True,0.335
